In [ ]:
import json
import os
from glob import glob
import pandas as pd
from IPython.display import display, Markdown

# Find the latest experiment summary JSON
json_files = sorted(glob("../experiment_results/experiment_summary_*.json"))
latest_file = json_files[-1]
print(f"Using: {latest_file}")

with open(latest_file) as f:
    data = json.load(f)

# Mapping: internal method name -> display name
method_map = {
    "dist_spec": "DSD",
    "dist_split_spec": "DSSD",
    "uncertainty_decoding": "CUHLM",
    "adaptive_tridecoding": r"CEE-SD",
}

# Mapping: internal model name -> display name
model_map = {
    "llama-2-13b": "Llama",
    "Qwen/Qwen1.5-7B-Chat": "Qwen1.5",
    "Qwen/Qwen3-14B": "Qwen3",
}

# Mapping: dataset path fragment -> display name
dataset_map = {
    "mt_bench": "MTBench",
    "gsm8k": "GSM8K",
    "humaneval": "HumanEval",
}

method_order = ["DSD", "DSSD", "CUHLM", r"CEE-SD"]
dataset_order = ["MTBench", "GSM8K", "HumanEval"]


def get_dataset_name(dataset_path):
    for key, name in dataset_map.items():
        if key in dataset_path.lower():
            return name
    return dataset_path


# Build lookup: (model, method, dataset) -> dict of metrics
lookup = {}
for item in data:
    if item.get("status") != "success":
        continue
    cfg = item["config"]
    res = item["result"]
    mode = cfg.get("eval_mode", "")
    if mode not in method_map:
        continue
    method = method_map[mode]
    model = model_map.get(cfg.get("target_model", ""), "")
    if not model:
        continue
    dataset = get_dataset_name(cfg.get("eval_dataset", ""))
    if not dataset:
        continue

    gen_tokens = res.get("generated_tokens", 0)
    if gen_tokens <= 0:
        continue

    # Thr: end-to-end token throughput
    thr = res.get("throughput", 0)

    # C_comm: WAN communication data per token (bytes)
    edge_cloud_bytes = res.get("edge_cloud_data_bytes", 0)
    ec_mb = round(edge_cloud_bytes / (1024 * 1024), 2)
    c_comm = round(ec_mb * 1024 * 1024 / gen_tokens, 2)

    # R_acce: acceptance rate of cloud LLM (%)
    draft_acc = res.get("draft_accepted_tokens", 0)
    draft_gen = res.get("draft_generated_tokens", 0)
    if mode == "uncertainty_decoding":
        r_acce = None
    elif draft_gen > 0:
        r_acce = round(draft_acc / draft_gen * 100, 2)
    else:
        r_acce = None

    # T_comm: communication time per token (ms)
    comm_time = res.get("communication_time", 0)
    t_comm = round(comm_time / gen_tokens * 1000, 1)

    # Fwd: forward times of cloud LLM
    fwd = int(res.get("target_forward_times", 0))

    # Cost: cost of inference
    draft_wall_time = res.get("draft_wall_time", 0.0)
    target_wall_time = res.get("target_wall_time", 0.0)
    cost = (
        4.4 * res.get("wall_time") / 3600
        if cfg.get("method") == "adaptive_tridecoding"
        or ("CEE" in method)
        or ("cee" in method)
        else 4.05 * res.get("wall_time") / 3600
    )

    lookup[(model, method, dataset)] = {
        "thr": thr,
        "c_comm": c_comm,
        "r_acce": r_acce,
        "t_comm": t_comm,
        "fwd": fwd,
        "cost": cost,
        # Per-model computation time (new fields)
        "comp_little": res.get("little_computation_time", 0),
        "comp_draft": res.get("draft_computation_time", 0),
        "comp_target": res.get("target_computation_time", 0),
    }

# --- Table 1: Main metrics ---
rows = []
for model_display in ["Llama", "Qwen1.5", "Qwen3"]:
    for method in method_order:
        row = {"Model": model_display, "Method": method}
        for dataset in dataset_order:
            key = (model_display, method, dataset)
            d = lookup.get(key)
            if d:
                row[f"{dataset}_Thr"] = f"{d['thr']:.2f}"
                row[f"{dataset}_CostDisplay"] = f"{d['cost']}"
                row[f"{dataset}_Racce"] = (
                    f"{d['r_acce']:.2f}" if d["r_acce"] is not None else "-"
                )
                row[f"{dataset}_Tcomm"] = f"{d['t_comm']:.2f}"
                row[f"{dataset}_Fwd"] = f"{d['fwd']}"
            else:
                for suffix in ["Thr", "CostDisplay", "Racce", "Tcomm", "Fwd"]:
                    row[f"{dataset}_{suffix}"] = "N/A"
        rows.append(row)

cols = pd.MultiIndex.from_tuples(
    [
        ("", "Model"),
        ("", "Method"),
        ("MTBench", "Thr."),
        ("MTBench", "cost"),
        ("MTBench", "$R_{acce}$"),
        ("MTBench", "$T_{comm}$"),
        ("MTBench", "Fwd."),
        ("GSM8K", "Thr."),
        ("GSM8K", "cost"),
        ("GSM8K", "$R_{acce}$"),
        ("GSM8K", "$T_{comm}$"),
        ("GSM8K", "Fwd."),
        ("HumanEval", "Thr."),
        ("HumanEval", "cost"),
        ("HumanEval", "$R_{acce}$"),
        ("HumanEval", "$T_{comm}$"),
        ("HumanEval", "Fwd."),
    ]
)

vals = []
for row in rows:
    vals.append(
        [
            row["Model"],
            row["Method"],
            row["MTBench_Thr"],
            row["MTBench_CostDisplay"],
            row["MTBench_Racce"],
            row["MTBench_Tcomm"],
            row["MTBench_Fwd"],
            row["GSM8K_Thr"],
            row["GSM8K_CostDisplay"],
            row["GSM8K_Racce"],
            row["GSM8K_Tcomm"],
            row["GSM8K_Fwd"],
            row["HumanEval_Thr"],
            row["HumanEval_CostDisplay"],
            row["HumanEval_Racce"],
            row["HumanEval_Tcomm"],
            row["HumanEval_Fwd"],
        ]
    )

df = pd.DataFrame(vals, columns=cols)

metric_directions = {
    "Thr.": True,
    "cost": False,
    "$R_{acce}$": True,
    "$T_{comm}$": False,
    "Fwd.": False,
}


def highlight_top_metrics(dataframe):
    styles = pd.DataFrame("", index=dataframe.index, columns=dataframe.columns)
    for model in dataframe[("", "Model")].dropna().unique():
        model_rows = dataframe[("", "Model")] == model
        for dataset in dataset_order:
            for metric, higher_is_better in metric_directions.items():
                col = (dataset, metric)
                values = pd.to_numeric(dataframe.loc[model_rows, col], errors="coerce")
                if values.notna().any():
                    ranked_values = values.dropna().drop_duplicates().sort_values(ascending=not higher_is_better)
                    best = ranked_values.iloc[0]
                    styles.loc[values[values == best].index, col] = "font-weight: bold"
                    if len(ranked_values) > 1:
                        second_best = ranked_values.iloc[1]
                        styles.loc[values[values == second_best].index, col] = "text-decoration: underline"
    return styles


display(Markdown("# Main Experiment Results"))
display(df.style.apply(highlight_top_metrics, axis=None))

def dataframe_to_markdown(dataframe):
    headers = [
        col[1] if not col[0] else f"{col[0]} {col[1]}"
        for col in dataframe.columns
    ]

    def escape_cell(value):
        return str(value).replace("|", r"\|")

    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join(["---"] * len(headers)) + " |",
    ]
    for _, row in dataframe.iterrows():
        lines.append("| " + " | ".join(escape_cell(value) for value in row) + " |")
    return "\n".join(lines)

markdown_table = dataframe_to_markdown(df)
display(Markdown("## Main Experiment Results (Markdown Source)"))
print(markdown_table)

BS = chr(92)

def latex_metric_text(value, kind):
    if value is None:
        return "-"
    if kind == "int":
        return f"{round(value):.0f}"
    return f"{value:.2f}"

def latex_ranked_text(value, best, second, kind):
    text = latex_metric_text(value, kind)
    if value is None:
        return text
    if value == best:
        return f"{BS}{BS}textbf{{{text}}}"
    if value == second:
        return f"{BS}{BS}underline{{{text}}}"
    return text

main_latex_metrics = [
    ("thr", True, "float"),
    ("cost", False, "float"),
    ("r_acce", True, "float"),
    ("t_comm", False, "float"),
    ("fwd", False, "int"),
]
rank_lookup = {}
for model in ["Llama", "Qwen1.5", "Qwen3"]:
    for dataset in dataset_order:
        for metric, higher_is_better, _ in main_latex_metrics:
            values = [
                lookup.get((model, method, dataset), {}).get(metric)
                for method in method_order
            ]
            values = [value for value in values if value is not None]
            ranked = sorted(set(values), reverse=higher_is_better)
            rank_lookup[(model, dataset, metric)] = (
                ranked[0] if ranked else None,
                ranked[1] if len(ranked) > 1 else None,
            )

main_latex_lines = [
    f"{BS}begin{{table*}}[!t]",
    f"{BS}centering",
    f"{BS}caption{{Detailed methods performance comparison across three datasets and three models. {{Thr., Cost, $R_{{acce}}$, $T_{{comm}}$, and Fwd. stand for the end-to-end token throughput, total cost per token, acceptance rate of cloud LLM, communication time per token, and forward times of cloud LLM. {BS}textit{{Arrows (${BS}downarrow/${BS}uparrow$) indicate better performance. Best results in {BS}textbf{{bold}}, second in {BS}underline{{underlined}}.}}}} }}",
    f"{BS}label{{tab:main_experiment_detail}}",
    f"{BS}resizebox{{0.9{BS}textwidth}}{{!}}{{%",
    f"{BS}begin{{tabular}}{{@{{}}ll ccccc ccccc ccccc@{{}}}}",
    f"{BS}toprule",
    f"{BS}multirow{{2}}{{*}}{{{BS}textbf{{Model}}}} & {BS}multirow{{2}}{{*}}{{{BS}textbf{{Method}}}} & {BS}multicolumn{{5}}{{c}}{{{BS}textbf{{MTBench}}}} & {BS}multicolumn{{5}}{{c}}{{{BS}textbf{{GSM8K}}}} & {BS}multicolumn{{5}}{{c}}{{{BS}textbf{{HumanEval}}}} {BS}{BS}",
    f"{BS}cmidrule(lr){{3-7}} {BS}cmidrule(lr){{8-12}} {BS}cmidrule(lr){{13-17}}",
    f"& & Thr. & Cost & $R_{{acce}}$ & $T_{{comm}}$ & Fwd. & Thr. & Cost & $R_{{acce}}$ & $T_{{comm}}$ & Fwd. & Thr. & Cost & $R_{{acce}}$ & $T_{{comm}}$ & Fwd. {BS}{BS}",
    f"& & (${BS}uparrow$) & (${BS}downarrow$) & (${BS}% {BS}uparrow$) & (${BS}downarrow$) & (${BS}# {BS}downarrow$) & (${BS}uparrow$) & (${BS}downarrow$) & (${BS}% {BS}uparrow$) & (${BS}downarrow$) & (${BS}# {BS}downarrow$) & (${BS}uparrow$) & (${BS}downarrow$) & (${BS}% {BS}uparrow$) & (${BS}downarrow$) & (${BS}# {BS}downarrow$) {BS}{BS} {BS}midrule",
]
for model in ["Llama", "Qwen1.5", "Qwen3"]:
    main_latex_lines.append(f"% ==================== {model} ====================")
    for method_index, method in enumerate(method_order):
        parts = []
        for dataset in dataset_order:
            for metric, _, kind in main_latex_metrics:
                value = lookup.get((model, method, dataset), {}).get(metric)
                best, second = rank_lookup[(model, dataset, metric)]
                parts.append(latex_ranked_text(value, best, second, kind))
        model_cell = f"{BS}multirow{{4}}{{*}}{{{BS}texttt{{{model}}}}}" if method_index == 0 else ""
        method_cell = f"{BS}Name{{}}" if method == "CEE-SD" else method
        prefix = f"{BS}rowcolor{{mycolor}} {BS}multicolumn{{1}}{{l}}{{{BS}cellcolor{{white}}}}" if method == "CEE-SD" else model_cell
        line = f"{prefix} & {method_cell} & " + " & ".join(parts) + f" {BS}{BS}"
        if method == "CEE-SD" and model != "Qwen3":
            line += f" {BS}midrule"
        main_latex_lines.append(line)
main_latex_lines.extend([
    f"{BS}bottomrule",
    f"{BS}end{{tabular}}%",
    "}",
    f"{BS}end{{table*}}",
])
main_latex_table = "\n".join(main_latex_lines)
display(Markdown("## Main Experiment Results (LaTeX Source)"))
print(main_latex_table)

# --- Table 2: Per-Model Computation Time ---
timing_rows = []
for model_display in ["Llama", "Qwen1.5", "Qwen3"]:
    for method in method_order:
        row = {"Model": model_display, "Method": method}
        for dataset in dataset_order:
            key = (model_display, method, dataset)
            d = lookup.get(key, {})
            if d:
                row[f"{dataset}_CompL"] = f"{d['comp_little']:.2f}"
                row[f"{dataset}_CompD"] = f"{d['comp_draft']:.2f}"
                row[f"{dataset}_CompT"] = f"{d['comp_target']:.2f}"
            else:
                for s in ["CompL", "CompD", "CompT"]:
                    row[f"{dataset}_{s}"] = "N/A"
        timing_rows.append(row)

timing_cols = pd.MultiIndex.from_tuples(
    [
        ("", "Model"),
        ("", "Method"),
        ("MTBench", "L.Comp(s)"),
        ("MTBench", "D.Comp(s)"),
        ("MTBench", "T.Comp(s)"),
        ("GSM8K", "L.Comp(s)"),
        ("GSM8K", "D.Comp(s)"),
        ("GSM8K", "T.Comp(s)"),
        ("HumanEval", "L.Comp(s)"),
        ("HumanEval", "D.Comp(s)"),
        ("HumanEval", "T.Comp(s)"),
    ]
)

timing_vals = []
for row in timing_rows:
    timing_vals.append(
        [
            row["Model"],
            row["Method"],
            row["MTBench_CompL"],
            row["MTBench_CompD"],
            row["MTBench_CompT"],
            row["GSM8K_CompL"],
            row["GSM8K_CompD"],
            row["GSM8K_CompT"],
            row["HumanEval_CompL"],
            row["HumanEval_CompD"],
            row["HumanEval_CompT"],
        ]
    )

timing_df = pd.DataFrame(timing_vals, columns=timing_cols)
display(
    Markdown(
        "## Per-Model Computation Time (s)\n*L=Little, D=Draft, T=Target. Requires re-running experiments.*"
    )
)
display(timing_df)

# Print LaTeX table rows
print("\n" + "=" * 80)
for row in rows:
    model = row["Model"]
    method = row["Method"]
    parts = [model, method]
    for ds in ["MTBench", "GSM8K", "HumanEval"]:
        parts.append(row[f"{ds}_Thr"])
        parts.append(row[f"{ds}_CostDisplay"])
        parts.append(row[f"{ds}_Racce"])
        parts.append(row[f"{ds}_Tcomm"])
        parts.append(row[f"{ds}_Fwd"])
    print(" & ".join(parts) + " \\\\")

In [ ]:
from IPython.display import HTML

display(Markdown("## Llama CEE Enhancement Table"))

cee_method_map = {
    "DSD": "CEE-DSD",
    "DSSD": "CEE-DSSD",
    "CUHLM": "CEE-CUHLM",
}
cee_mode_map = {
    "cee_dsd": "CEE-DSD",
    "cee_dssd": "CEE-DSSD",
    "cee_cuhlm": "CEE-CUHLM",
}

cee_lookup = dict(lookup)
for item in data:
    if item.get("status") != "success":
        continue
    cfg = item["config"]
    mode = cfg.get("eval_mode", "")
    if mode not in cee_mode_map:
        continue
    model = model_map.get(cfg.get("target_model", ""), "")
    dataset = get_dataset_name(cfg.get("eval_dataset", ""))
    res = item["result"]
    gen_tokens = res.get("generated_tokens", 0)
    if not model or not dataset or gen_tokens <= 0:
        continue
    draft_gen = res.get("draft_generated_tokens", 0)
    r_acce = None if mode == "cee_cuhlm" or draft_gen <= 0 else round(res.get("draft_accepted_tokens", 0) / draft_gen * 100, 2)
    cost = (
        4.4 * res.get("wall_time") / 3600
        if ("CEE" in cee_mode_map[mode]) or ("cee" in mode)
        else 4.05 * res.get("wall_time") / 3600
    )
    cee_lookup[(model, cee_mode_map[mode], dataset)] = {
        "thr": res.get("throughput", 0),
        "cost": cost,
        "r_acce": r_acce,
        "t_comm": round(res.get("communication_time", 0) / gen_tokens * 1000, 1),
        "fwd": int(res.get("target_forward_times", 0)),
    }

metrics = [
    ("thr", "Thr. (&uarr;)", True, "float"),
    ("cost", "Cost (&darr;)", False, "float"),
    ("r_acce", "R<sub>acce</sub> (% &uarr;)", True, "float"),
    ("t_comm", "T<sub>comm</sub> (&darr;)", False, "float"),
    ("fwd", "Fwd. (# &darr;)", False, "int"),
]

def metric_average(model, method, metric):
    values = []
    for dataset in dataset_order:
        value = cee_lookup.get((model, method, dataset), {}).get(metric)
        if value is not None:
            values.append(value)
    return sum(values) / len(values) if values else None

def fmt_value(value, kind):
    if value is None:
        return "-"
    if kind == "int":
        return f"{round(value):.0f}"
    if kind == "float3":
        return f"{value:.3f}"
    return f"{value:.2f}"

def fmt_delta(original, enhanced, higher_is_better):
    if original is None or enhanced is None or original == 0:
        return "-"
    delta = (enhanced - original) / original if higher_is_better else (original - enhanced) / original
    sign = "+" if delta >= 0 else ""
    return f"<strong>{sign}{delta * 100:.0f}%</strong>" if delta >= 0 else f"{delta * 100:.0f}%"

def fmt_pair(original, enhanced, higher_is_better, kind):
    original_text = fmt_value(original, kind)
    enhanced_text = fmt_value(enhanced, kind)
    if original is not None and enhanced is not None:
        if enhanced > original if higher_is_better else enhanced < original:
            enhanced_text = f"<u>{enhanced_text}</u>"
        elif original > enhanced if higher_is_better else original < enhanced:
            original_text = f"<u>{original_text}</u>"
    return f"{original_text}, {enhanced_text}"

html_rows = []
for method, cee_method in cee_method_map.items():
    value_cells = []
    delta_cells = []
    for metric, _, higher_is_better, kind in metrics:
        original = metric_average("Llama", method, metric)
        enhanced = metric_average("Llama", cee_method, metric)
        value_cells.append(f"<td>{fmt_pair(original, enhanced, higher_is_better, kind)}</td>")
        delta_cells.append(f"<td>{fmt_delta(original, enhanced, higher_is_better)}</td>")
    html_rows.append(f"<tr><td rowspan='2'>{method}</td>{''.join(value_cells)}</tr>")
    html_rows.append(f"<tr>{''.join(delta_cells)}</tr>")

ours_cells = []
for metric, _, _, kind in metrics:
    value = metric_average("Llama", "CEE-SD", metric)
    text = fmt_value(value, kind)
    ours_cells.append(f"<td><strong>{text}</strong></td>")
html_rows.append(f"<tr style='background-color: #eef4ff;'><td>CEE-SD</td>{''.join(ours_cells)}</tr>")

headers = "".join(f"<th>{label}</th>" for _, label, _, _ in metrics)
display(HTML(f"""
<table>
  <thead>
    <tr><th rowspan='2'>Method</th><th colspan='5'>Performance</th></tr>
    <tr>{headers}</tr>
  </thead>
  <tbody>
    {''.join(html_rows)}
  </tbody>
</table>
"""))

markdown_headers = [
    "Method",
    "Thr. (↑)",
    "Cost (↓)",
    "R_acce (% ↑)",
    "T_comm (↓)",
    "Fwd. (# ↓)",
]
markdown_rows = []
for method, cee_method in cee_method_map.items():
    value_row = [method]
    delta_row = ["Delta"]
    for metric, _, higher_is_better, kind in metrics:
        original = metric_average("Llama", method, metric)
        enhanced = metric_average("Llama", cee_method, metric)
        value_row.append(fmt_pair(original, enhanced, higher_is_better, kind))
        delta_row.append(fmt_delta(original, enhanced, higher_is_better))
    markdown_rows.append(value_row)
    markdown_rows.append(delta_row)

ours_row = ["CEE-SD"]
for metric, _, _, kind in metrics:
    ours_row.append(f"**{fmt_value(metric_average('Llama', 'CEE-SD', metric), kind)}**")
markdown_rows.append(ours_row)

def escape_markdown_cell(value):
    return str(value).replace("|", r"\|")

llama_cee_markdown_table = "\n".join([
    "| " + " | ".join(markdown_headers) + " |",
    "| " + " | ".join(["---"] * len(markdown_headers)) + " |",
    *("| " + " | ".join(escape_markdown_cell(value) for value in row) + " |" for row in markdown_rows),
])
display(Markdown("## Llama CEE Enhancement Table (Markdown Source)"))
print(llama_cee_markdown_table)

def latex_clean(value):
    return str(value).replace("<strong>", r"\\textbf{").replace("</strong>", "}").replace("<u>", r"\\underline{").replace("</u>", "}").replace("%", r"\%")

cee_latex_lines = [
    r"\begin{table}[!t]",
    r"\centering",
    r"\caption{Enhancement of our CEE architecture for baselines on Llama. Data format is presented as: Absolute values (Original, +CEE) (top) and relative improvement $\Delta$ (bottom).}",
    r"\label{tab:main_experiment_average}",
    r"\resizebox{\linewidth}{!}{%",
    r"\setlength{\tabcolsep}{3.5pt}",
    r"\begin{tabular}{@{}l ccccc@{}}",
    r"\toprule",
    r'\multirow{2}{*}{\textbf{Method}} & \multicolumn{5}{c}{\textbf{Performance}} \\',
    r"\cmidrule(lr){2-6}",
    r'& Thr. ($\uparrow$) & Cost ($\downarrow$) & $R_{acce}$ ($\% \uparrow$) & $T_{comm}$ ($\downarrow$) & Fwd. ($\# \downarrow$) \\ \midrule',
]
for method, cee_method in cee_method_map.items():
    value_parts = []
    delta_parts = []
    for metric, _, higher_is_better, kind in metrics:
        original = metric_average("Llama", method, metric)
        enhanced = metric_average("Llama", cee_method, metric)
        value_parts.append(latex_clean(fmt_pair(original, enhanced, higher_is_better, kind)))
        delta_parts.append(f"({latex_clean(fmt_delta(original, enhanced, higher_is_better))})")
    cee_latex_lines.append(f"\\multirow{{2}}{{*}}{{{method}}} & " + " & ".join(value_parts) + r' \\')
    cee_latex_lines.append(" & " + " & ".join(delta_parts) + r" \\ \addlinespace")
ours_parts = []
for metric, _, _, kind in metrics:
    ours_parts.append(f"\\textbf{{{fmt_value(metric_average('Llama', 'CEE-SD', metric), kind)}}}")
cee_latex_lines.append(r"\rowcolor{mycolor}")
cee_latex_lines.append(r"\Name{} & " + " & ".join(ours_parts) + r" \\ \bottomrule")
cee_latex_lines.extend([
    r"\end{tabular}",
    r"}",
    r"\end{table}",
])
llama_cee_latex_table = "\n".join(cee_latex_lines)
display(Markdown("## Llama CEE Enhancement Table (LaTeX Source)"))
print(llama_cee_latex_table)
